# Lecture 6 Lab: Chunking a Real PDF
**Vector Databases Course** | Estimated time: 75 minutes | Platform: Google Colab (free)

**Goal:** download a real research paper as a PDF, inspect and clean the extracted text, and cut it into chunks four different ways. You will then attach metadata and stable IDs, and measure which strategy retrieves the right passage most often.

- Extract and clean text from a PDF, and see what PDF extraction gets wrong.
- Implement fixed-size, recursive, sentence-window and semantic chunking.
- Check chunk lengths in tokens against the model's input limit.
- Build chunk records with deterministic IDs, content hashes and page metadata.
- Evaluate strategies and sizes with a hit rate at 3 on eight questions.

Run each cell in order. Write your answers in the **YOUR TURN** answer cells.

## Setup
### Cell 1: install and import

In [ ]:
!pip install -q pypdf requests sentence-transformers numpy pandas matplotlib
import re, json, hashlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import requests
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

## Part 1: Get the text out of a PDF
We use the 2017 paper 'Attention Is All You Need' from arXiv. If the download is blocked, upload any text-heavy PDF named `paper.pdf` (Colab's file panel, left sidebar), and adapt the questions in Cell 9.

### Cell 2: download and extract

In [ ]:
import os
url = "https://arxiv.org/pdf/1706.03762"
if not os.path.exists("paper.pdf"):   # skip the download if you uploaded your own paper.pdf
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
    assert r.content[:4] == b"%PDF", "Download did not return a PDF: upload your own paper.pdf instead."
    open("paper.pdf", "wb").write(r.content)

reader = PdfReader("paper.pdf")
pages = [p.extract_text() or "" for p in reader.pages]
print(len(pages), "pages,", sum(len(p) for p in pages), "characters")
print(pages[2][:700])

> **YOUR TURN:** Read the extracted text. List three things that would hurt retrieval (for example broken words, repeated headers, equations or table noise).

1. *(write here)*
2. *(write here)*
3. *(write here)*

### Cell 3: clean the text

In [ ]:
def clean(text):
    text = re.sub(r"-\n(?=[a-z])", "", text)        # rejoin hyphenated line breaks
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)    # drop lines with only a number
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

doc_text = "\n".join(clean(p) for p in pages)
flat = re.sub(r"\s+", " ", doc_text)   # one long line for sentence tools
print(len(doc_text), len(flat))

## Part 2: Four chunkers
### Cell 4: fixed size with overlap, and recursive

In [ ]:
def fixed_chunks(text, size=800, overlap=150):
    step = size - overlap
    return [text[i:i + size] for i in range(0, max(len(text) - overlap, 1), step)]

def recursive_chunks(text, size=800, seps=("\n\n", "\n", ". ", " ")):
    if len(text) <= size:
        return [text]
    for k, sep in enumerate(seps):
        if sep in text:
            parts = [p + sep for p in text.split(sep)]
            parts[-1] = parts[-1][:-len(sep)]
            rest = seps[k + 1:]
            break
    else:
        return [text[i:i + size] for i in range(0, len(text), size)]
    chunks, cur = [], ""
    for part in parts:
        if len(part) > size:
            if cur:
                chunks.append(cur); cur = ""
            chunks.extend(recursive_chunks(part, size, rest))
        elif len(cur) + len(part) <= size:
            cur += part
        else:
            chunks.append(cur); cur = part
    if cur:
        chunks.append(cur)
    return [c.strip() for c in chunks if c.strip()]

### Cell 5: sentence-window and semantic

In [ ]:
def sentence_chunks(text, n=5, overlap=1):
    sents = re.split(r"(?<=[.!?])\s+", text)
    return [" ".join(sents[i:i + n]) for i in range(0, len(sents), n - overlap)]

def semantic_chunks(text, percentile=20, max_chars=1200):
    sents = [s for s in re.split(r"(?<=[.!?])\s+", text) if len(s) > 20]
    E = model.encode(sents, normalize_embeddings=True)
    sims = np.sum(E[:-1] * E[1:], axis=1)   # neighbour similarity
    cut = np.percentile(sims, percentile)
    chunks, cur = [], sents[0]
    for s, sim in zip(sents[1:], sims):
        if sim < cut or len(cur) + len(s) > max_chars:
            chunks.append(cur); cur = s
        else:
            cur += " " + s
    chunks.append(cur)
    return chunks

### Cell 6: run all four and compare

In [ ]:
strategies = {
    "fixed": fixed_chunks(flat),
    "recursive": recursive_chunks(doc_text),
    "sentence": sentence_chunks(flat),
    "semantic": semantic_chunks(flat),
}

def stats(chunks):
    lens = np.array([len(c) for c in chunks])
    ends = np.mean([c.rstrip()[-1:] in ".?!" for c in chunks])
    return {"chunks": len(chunks), "mean": int(lens.mean()), "min": int(lens.min()),
            "max": int(lens.max()), "end at sentence": round(float(ends), 2)}

pd.DataFrame({k: stats(v) for k, v in strategies.items()}).T

> **YOUR TURN:** Which strategy produces the most uniform chunk sizes, and which most often ends at a sentence boundary? Print three chunks from each strategy and describe one weakness you can see.

In [ ]:
for name, ch in strategies.items():
    print(f"\n########## {name} ##########")
    mid = len(ch) // 2
    for c in ch[mid:mid + 3]:
        print(f"[{len(c)} chars] {c}\n---")

**Most uniform sizes:** *(write here)*

**Most often ends at a sentence boundary:** *(write here)*

**One weakness per strategy:**

*(write here)*

### Cell 7: token lengths versus the model limit

In [ ]:
limit = model.max_seq_length
print("model limit:", limit, "tokens")
for name, ch in strategies.items():
    toks = [len(model.tokenizer(c)["input_ids"]) for c in ch]
    print(f"{name:10s} over limit: {sum(t > limit for t in toks):3d} of {len(ch)}"
          f"  max {max(toks)} tokens")

> **YOUR TURN:** Which chunks would be silently truncated by the model? What setting would you change for the fixed chunker so none exceed the limit? Check your answer by re-running.

In [ ]:
# Try a smaller size for the fixed chunker and re-check (edit the numbers)
new_size, new_overlap = 500, 75
fixed_small = fixed_chunks(flat, size=new_size, overlap=new_overlap)
toks = [len(model.tokenizer(c)["input_ids"]) for c in fixed_small]
print(f"size={new_size}: over limit {sum(t > limit for t in toks)} of {len(fixed_small)}, max {max(toks)} tokens")

**My answer:**

*(write here)*

## Part 3: Metadata and stable IDs
### Cell 8: build chunk records page by page

In [ ]:
doc_id = "attention-2017"
records = []
for pno, ptext in enumerate(pages, start=1):
    for j, c in enumerate(recursive_chunks(clean(ptext), size=800)):
        records.append({
            "id": f"{doc_id}:p{pno}:{j}",
            "text": c,
            "metadata": {"source": "paper.pdf", "page": pno, "chunk": j,
                         "sha1": hashlib.sha1(c.encode()).hexdigest()[:12]},
        })
print(len(records), "records")
print(json.dumps(records[10], indent=2)[:500])

with open("chunks.jsonl", "w") as f:
    for rec in records:
        f.write(json.dumps(rec) + "\n")

> **YOUR TURN:** Why include the content hash and the page number? Explain how the hash lets you skip unchanged pages on the next ingestion run, and what you would do for a page whose hash changed.

In [ ]:
# Simulated re-ingestion: hash each page, "edit" one page, and see which pages need re-embedding
def page_hash(text):
    return hashlib.sha1(clean(text).encode()).hexdigest()[:12]

stored = {f"{doc_id}:p{i}": page_hash(p) for i, p in enumerate(pages, start=1)}   # what the DB remembers

edited = list(pages)
edited[4] = edited[4] + " An extra sentence added in the new version."          # simulate an edit to page 5

to_reindex = [f"{doc_id}:p{i}" for i, p in enumerate(edited, start=1)
              if stored.get(f"{doc_id}:p{i}") != page_hash(p)]
print("pages to re-embed:", to_reindex, "| skipped:", len(pages) - len(to_reindex))

**My answer:**

*(write here)*

## Part 4: Which strategy retrieves best?
Each question comes with a key phrase that appears in the answer passage. A strategy gets a hit when one of the top three chunks contains the phrase. Hyphens and whitespace are ignored when matching, because PDF extraction breaks words unpredictably.

### Cell 9: evaluation set and hit rate

In [ ]:
def norm(s):
    return re.sub(r"[\s\-]", "", s.lower())

eval_set = [
    ("How is attention computed from queries, keys and values?",
     "scaled dot-product attention"),
    ("Why does the model use several attention heads?",
     "multi-head attention"),
    ("How does the model know word order?",
     "sinusoid"),
    ("Which optimizer was used for training?",
     "adam"),
    ("How many identical layers does the encoder have?",
     "identical layers"),
    ("Which data was used for English-German translation?",
     "wmt 2014"),
    ("What hardware was the model trained on?",
     "p100"),
    ("What BLEU score did the big model reach on English-to-German?",
     "28.4"),
]
print("answers present in text:", [norm(k) in norm(flat) for _, k in eval_set])

def hit_rate(chunks, k=3):
    E = model.encode(chunks, normalize_embeddings=True)
    hits = 0
    for q, key in eval_set:
        qv = model.encode(q, normalize_embeddings=True)
        top = np.argsort(-(E @ qv))[:k]
        hits += any(norm(key) in norm(chunks[i]) for i in top)
    return hits / len(eval_set)

scores = {name: round(hit_rate(ch), 2) for name, ch in strategies.items()}
print(scores)
pd.Series(scores).plot.bar(ylim=(0, 1), title="hit rate @ 3"); plt.show()

> **YOUR TURN:** If any value in the 'answers present' list is False, fix the key phrase before trusting the result. Then report the hit rate per strategy and the question each strategy misses.

| Strategy | Chunks | Hit rate @ 3 | A question it misses |
|---|---|---|---|
| fixed | | | |
| recursive | | | |
| sentence | | | |
| semantic | | | |

In [ ]:
# Helper: which questions does each strategy miss? (fills in the table above)
def missed(chunks, k=3):
    E = model.encode(chunks, normalize_embeddings=True)
    out = []
    for q, key in eval_set:
        qv = model.encode(q, normalize_embeddings=True)
        top = np.argsort(-(E @ qv))[:k]
        if not any(norm(key) in norm(chunks[i]) for i in top):
            out.append(q)
    return out

table = []
for name, ch in strategies.items():
    m = missed(ch)
    table.append({"strategy": name, "chunks": len(ch), "hit@3": scores[name],
                  "misses": " | ".join(m) if m else "none"})
pd.set_option("display.max_colwidth", 200)
pd.DataFrame(table)

## Part 5: Sweep size and overlap
### Cell 10: grid over chunk size and overlap

In [ ]:
rows = []
for size in [300, 600, 1200]:
    for ov in [0.0, 0.15]:
        ch = fixed_chunks(flat, size=size, overlap=int(size * ov))
        rows.append({"size": size, "overlap": ov, "chunks": len(ch),
                     "hit@3": round(hit_rate(ch), 2)})
pd.DataFrame(rows).pivot(index="size", columns="overlap", values="hit@3")

> **YOUR TURN:** Which size and overlap work best? With only eight questions, how confident are you? Write two new questions in the same format and see whether the ranking changes.

In [ ]:
# Add two questions of your own (check the key phrase appears in the paper first!)
extra = [
    ("your new question 1", "key phrase 1"),
    ("your new question 2", "key phrase 2"),
]
print("answers present:", [norm(k) in norm(flat) for _, k in extra])

original_eval = list(eval_set)
eval_set = original_eval + extra          # hit_rate reads the global eval_set
rows2 = []
for size in [300, 600, 1200]:
    for ov in [0.0, 0.15]:
        ch = fixed_chunks(flat, size=size, overlap=int(size * ov))
        rows2.append({"size": size, "overlap": ov, "hit@3": round(hit_rate(ch), 2)})
print(pd.DataFrame(rows2).pivot(index="size", columns="overlap", values="hit@3"))
eval_set = original_eval                  # restore

**My answer:**

*(write here)*

## Part 6: Inspect the worst chunks
### Cell 11: find noisy chunks

In [ ]:
def noise(c):
    letters = sum(ch.isalpha() for ch in c)
    return 1 - letters / max(len(c), 1)

ranked = sorted(strategies["recursive"], key=noise, reverse=True)
for c in ranked[:3]:
    print(round(noise(c), 2), "|", c[:200].replace("\n", " "), "\n")

> **YOUR TURN:** What do the noisiest chunks contain (tables, equations, references)? Decide whether to drop, clean or keep them, and write the rule you would add to `clean()`.

**What they contain:** *(write here)*

**Decision and rule:** *(write here)*

## Reflection questions

**1. Which chunk size gave the best hit rate, and why might it differ for a legal contract or a FAQ?**

*(write here)*

**2. Semantic chunking costs an embedding per sentence. Under what conditions would you pay for it?**

*(write here)*

**3. A document is edited and re-uploaded. Describe exactly how your IDs and hashes prevent duplicates.**

*(write here)*

**4. Your evaluation uses eight questions written by you. List two ways this could mislead you.**

*(write here)*

---
## Deliverable
Submit the executed notebook, the completed table, `chunks.jsonl`, the answers to each YOUR TURN box, and short answers to the reflection questions.

## Troubleshooting
- **Download fails or returns HTML:** upload your own PDF as `paper.pdf` and replace the questions in Cell 9.
- **Extracted text looks garbled:** the PDF may be scanned; use OCR first.
- **All hit rates are zero or the run is slow:** check the 'answers present' list, and reduce the text with `pages[:6]` on slow CPUs.